# Test set + transfer eval: base vs GRPO (lr 1e-4, seed 1 / seed 2)

The eval for the final report. Hyperparameters were already picked on the validation set (lr 1e-4, 1250 steps), so **these results are for reporting only. Don't use them to tune anything or pick checkpoints**.

| Dataset | Source | Questions | Tag in result files |
|---|---|---|---|
| MedQA test set | `GBaker/MedQA-USMLE-4-options` test split | 1273 | `medqa4` |
| MedMCQA (transfer) | `openlifescienceai/medmcqa` validation (the test set has no public answers) | about 4183 | `medmcqa` |
| PubMedQA (transfer) | `qiaojin/PubMedQA` pqa_labeled (expert-labeled) | 1000 | `pubmedqa` |

The three eval modes are exactly the same as in the validation notebook (all models in **bf16, no quantization**):

| Mode | Script | confidence |
|---|---|---|
| `direct` | `eval/evaluate.py --quantization none` | letter probabilities at the first token |
| `cot` | `eval/cot_eval.py` (GRPO training prompt, greedy) | probability of the letter token after `Answer:` |
| `cot_vote8` | same, sampled 8 times (temperature 1.0) | share of the 8 samples that picked this answer |

PubMedQA CoT prompt: `Context:` + the abstract goes before the question and the last sentence becomes `X is one of A, B, or C`; everything else matches the training prompt. The MedQA / MedMCQA prompts are word for word the same as in training.

Run in order: check GPU -> set variables -> pull the repo -> install dependencies -> (if pip asks) restart session -> run from cell 2 (set variables) down, skipping cell 4. After the restart cell 3 is optional: cell 5 does its own `import os, Path` and changes into the repo folder, and later cells don't use the `sh()` from cell 3. Use an **A100**.

Time (rough guess from about 5 min/model on the 500 validation questions, not measured): CoT about 40-60 minutes per model, about 2-3 hours for all 3; direct about 10 minutes total. You can leave just `'medqa'` in `DATASETS` in cell 2 to run MedQA first.
Results go to `RESULTS_DIR` on Drive; finished evals get skipped automatically, so after a disconnect just start again from cell 1.

In [ ]:
# 1. Check the GPU and Python version
!nvidia-smi
import sys, torch
print('Python', sys.version.split()[0])
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print('GPU:', torch.cuda.get_device_name(0), f'| compute {major}.{minor} →', 'bf16' if major >= 8 else 'fp16')
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'
assert sys.version_info >= (3, 11), 'TRL 1.16 needs Python >= 3.11'

In [ ]:
# 2. Set variables
REPO_URL = 'https://github.com/skjdsk/medqa-calibration.git'
BRANCH   = 'main'
REPO_DIR = '/content/medqa-calibration'
TRL_VERSION = '1.16.0'
USE_VLLM = True  # cell 4 installs the same vLLM version as training

DRIVE_ROOT = '/content/drive/MyDrive/medqa-calibration'
# adapters to evaluate: {model name: adapter folder}; base doesn't need one
ADAPTERS = {
    'grpo_lr1e-4_seed1': f'{DRIVE_ROOT}/grpo/grpo_train5000_g8_ep1_len1024_lr0.0001_seed1_bf16/adapter',
    'grpo_lr1e-4_seed2': f'{DRIVE_ROOT}/grpo/grpo_train5000_g8_ep1_len1024_lr0.0001_seed2_bf16/adapter',
}
RESULTS_DIR = f'{DRIVE_ROOT}/eval_test_grpo'   # kept apart from the validation results (eval_val)
DATASETS    = ['medqa', 'medmcqa', 'pubmedqa'] # change to ['medqa'] to run just the MedQA test set first
N_SAMPLES   = 8      # samples for cot_vote (matches the 8 answers per question in training)
N_DEBUG     = None   # set to 20 to try a few questions first (they go in the same folder, so delete those CSVs before the real run); use None for real

In [ ]:
# 3. git clone / pull repo
# Private repo: add GITHUB_TOKEN (a GitHub personal access token) under the key icon (Secrets) on the left in Colab, and turn on "Notebook access"
import os, subprocess
from pathlib import Path

def sh(cmd, shown=None):
    print('$', shown or cmd)
    subprocess.run(cmd, shell=True, check=True)

clone_url = REPO_URL
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
    if gh_token and REPO_URL.startswith('https://github.com/'):
        clone_url = REPO_URL.replace('https://', f'https://{gh_token}@')
except Exception:
    pass  # no GITHUB_TOKEN set, treat it as a public repo

if (Path(REPO_DIR) / '.git').exists():
    sh(f'git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {clone_url} {REPO_DIR}', shown=f'git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
sh('git log -1 --oneline')
for f in ['eval/evaluate.py', 'eval/cot_eval.py', 'eval/metrics.py', 'train/grpo/grpo_data.py', 'train/grpo/rewards.py', 'data/medqa_split_seed42.json']:
    assert Path(f).exists(), f'{f} not found in the repo: make sure it has been pushed to {BRANCH}'
print('cwd =', os.getcwd())

In [ ]:
# 4. Install dependencies
from pathlib import Path
ip = get_ipython()
if Path('requirements.txt').exists():
    ip.run_line_magic('pip', 'install -q -r requirements.txt')
ip.run_line_magic('pip', f'install -q "trl[peft,quantization]=={TRL_VERSION}" "matplotlib" "pandas" "pytest"')
if USE_VLLM:
    # vLLM installs the torch version it wants, so you usually need Runtime -> Restart session afterwards, then rerun from the "set variables" cell
    # vLLM 0.28+ dropped bitsandbytes quantization, but with QLoRA (4-bit) TRL has vLLM load the model through bitsandbytes
    # -> pinned to 0.27.1 (the last version with bitsandbytes, inside TRL 1.16's allowed 0.22-0.31 range, needs torch==2.13.0)
    ip.run_line_magic('pip', f'install -q "trl[vllm]=={TRL_VERSION}" "vllm==0.27.1"')

# Colab's preinstalled torchao 0.10 is too old: peft 0.21 errors out when adding LoRA without quantization (--quantization none)
# "only versions above 0.16.0 are supported". We don't use torchao (neither vllm nor peft needs it), so just uninstall it
ip.system('pip uninstall -y -q torchao')

import importlib.metadata as md
for pkg in ['torch', 'transformers', 'trl', 'peft', 'bitsandbytes', 'accelerate', 'datasets'] + (['vllm'] if USE_VLLM else []):
    try:
        print(f'{pkg:14s}', md.version(pkg))
    except md.PackageNotFoundError:
        print(f'{pkg:14s} not installed')
print('\nIf pip says a restart is needed above (or imports fail later): Runtime -> Restart session, then rerun from the "set variables" cell.')

**After installing: Runtime -> Restart session**, then run from cell 2 (set variables) down (skip cell 4).

In [ ]:
# 5. Mount Google Drive and check the adapters exist
import os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
os.chdir(REPO_DIR)
for name, path in ADAPTERS.items():
    ok = Path(path, 'adapter_config.json').exists()
    print(f'{name:12s}', '✅' if ok else '❌ not found', path)
    assert ok, f'adapter for {name} not found: check the path, or wait for Drive to finish syncing'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('results folder:', RESULTS_DIR)

In [ ]:
# 6. (Optional) set HF_TOKEN: Qwen2.5 and MedQA are public so downloads work without it, but with it they're more reliable and less likely to get rate-limited
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN set')
except Exception:
    print('No HF_TOKEN found (optional), downloading anonymously')

In [ ]:
# 7. Unit tests (a few seconds)
!python -m pytest -q eval/test_cot_eval.py eval/test_metrics.py train/grpo/test_rewards.py

In [ ]:
# 8. Helper for running evals: skips anything whose result file already exists (so reruns after a disconnect don't redo work)
import subprocess
N_ARG = f' --n {N_DEBUG}' if N_DEBUG else ''

def run_eval(cmd, expected_csv):
    if Path(RESULTS_DIR, expected_csv).exists():
        print('already there, skipping:', expected_csv)
        return
    print('$', cmd)
    get_ipython().system(f'{cmd} 2>&1 | tee -a {RESULTS_DIR}/eval_log.txt')
    assert Path(RESULTS_DIR, expected_csv).exists(), f'{expected_csv} was not created: check the error above'

MODELS = {'base': None, **ADAPTERS}
# dataset -> (command-line args, tag in the result file name); tags match the naming in evaluate.py / cot_eval.py
DS_ARGS = {
    'medqa':    ('--dataset medqa --medqa_subset test', 'medqa4'),
    'medmcqa':  ('--dataset medmcqa', 'medmcqa'),
    'pubmedqa': ('--dataset pubmedqa', 'pubmedqa'),
}
TAGS = [DS_ARGS[d][1] for d in DATASETS]
print('models:', list(MODELS)); print('datasets:', TAGS)

In [ ]:
# 9. Direct answer (letter-logit), one process per model x dataset; MedQA without --medqa_subset = the official test split
for name, adapter in MODELS.items():
    ad = f' --adapter {adapter}' if adapter else ''
    for ds in DATASETS:
        run_eval(f'python -m eval.evaluate --dataset {ds} --quantization none'
                 f'{ad} --model_name {name} --out_dir {RESULTS_DIR}{N_ARG}',
                 f'preds_{name}_{DS_ARGS[ds][1]}_direct.csv')

In [ ]:
# 10. Reason then answer (GRPO training prompt, vLLM). One process per model x dataset (vLLM only fully frees memory when it exits)
for ds in DATASETS:
    ds_arg, tag = DS_ARGS[ds]
    for name, adapter in MODELS.items():
        ad = f' --adapter {adapter}' if adapter else ''
        run_eval(f'python -m eval.cot_eval {ds_arg}{ad} --model_name {name} --n_samples {N_SAMPLES} '
                 f'--out_dir {RESULTS_DIR}{N_ARG}',
                 f'preds_{name}_{tag}_cot_vote{N_SAMPLES}.csv' if N_SAMPLES else f'preds_{name}_{tag}_cot.csv')

In [ ]:
# 11. Summary: accuracy, ECE, Brier, NLL, etc. (eval/metrics.py)
import pandas as pd
from IPython.display import display, Image

MODES = ['direct', 'cot', f'cot_vote{N_SAMPLES}']
# only summarize files that exist (so it doesn't break if some aren't done yet)
found = [str(f) for t in TAGS for m in MODES for f in sorted(Path(RESULTS_DIR).glob(f'preds_*_{t}_{m}.csv'))]
print(f'found {len(found)} result files')
files = ' '.join(f'"{f}"' for f in found)
!python -m eval.metrics --pred_csv {files} --out_dir {RESULTS_DIR}/metrics > /dev/null
s = pd.read_csv(f'{RESULTS_DIR}/metrics/summary.csv')
cols = ['dataset', 'eval_mode', 'model_name', 'n', 'accuracy', 'mean_confidence', 'ece', 'brier', 'nll',
        'hc_wrong_rate', 'mean_conf_wrong', 'mean_letter_mass']
display(s[cols].sort_values(['dataset', 'eval_mode', 'model_name']).round(4))
# note: NLL for cot_vote isn't reliable. If none of the 8 votes picked the right answer, p=0 gets clipped to 1e-12 and that question's NLL is 27.6

# extra info for cot: format, truncation, length; sample accuracy for cot_vote
for mode, extra in [('cot', ['format_ok', 'truncated', 'n_completion_tokens']), (f'cot_vote{N_SAMPLES}', ['sample_acc', 'format_ok'])]:
    paths = [Path(RESULTS_DIR, f'preds_{m}_{t}_{mode}.csv') for t in TAGS for m in MODELS]
    parts = [pd.read_csv(q) for q in paths if q.exists()]
    if parts:
        print(mode); display(pd.concat(parts).groupby(['dataset', 'model_name'])[extra].mean().round(4))

In [ ]:
# 12. Paired comparison with base (matched by question), for each GRPO model x dataset x eval mode
#   - questions fixed / broken, with a rough significance line of ±2 x sqrt(sum of both) (approximate sign test)
#   - paired bootstrap (resample questions with replacement 2000 times): 95% interval for the difference (GRPO - base) in accuracy, mean confidence and ECE
#     interval doesn't include 0 -> the difference is fairly reliable on this set of questions
import numpy as np
from eval.metrics import prob_matrix, expected_calibration_error

def conf_correct(df):
    probs, gold = prob_matrix(df)
    return probs.max(axis=1), (probs.argmax(axis=1) == gold).astype(float)

rng = np.random.default_rng(0)
rows = []
for t in TAGS:
    for mode in MODES:
        p = Path(RESULTS_DIR, f'preds_base_{t}_{mode}.csv')
        if not p.exists():
            continue
        b = pd.read_csv(p).set_index('question_id')
        cb, kb = conf_correct(b)
        for name in ADAPTERS:
            q = Path(RESULTS_DIR, f'preds_{name}_{t}_{mode}.csv')
            if not q.exists():
                continue
            g = pd.read_csv(q).set_index('question_id')
            assert set(g.index) == set(b.index), f'{name} {t} {mode} has different questions from base'
            g = g.loc[b.index]
            cg, kg = conf_correct(g)
            lost, gained = int(((kb == 1) & (kg == 0)).sum()), int(((kb == 0) & (kg == 1)).sum())
            n = len(b)
            boot = {'acc': [], 'conf': [], 'ece': []}
            for _ in range(2000):
                i = rng.integers(0, n, n)
                boot['acc'].append(kg[i].mean() - kb[i].mean())
                boot['conf'].append(cg[i].mean() - cb[i].mean())
                boot['ece'].append(expected_calibration_error(cg[i], kg[i], 15) - expected_calibration_error(cb[i], kb[i], 15))
            r = {'dataset': t, 'mode': mode, 'model': name, 'n': n, 'same_pred': round((b.pred == g.pred).mean(), 3),
                 'lost': lost, 'gained': gained, 'net': gained - lost, 'sig_line': round(2 * (lost + gained) ** 0.5)}
            for k, point in [('acc', kg.mean() - kb.mean()), ('conf', cg.mean() - cb.mean()),
                             ('ece', expected_calibration_error(cg, kg, 15) - expected_calibration_error(cb, kb, 15))]:
                lo, hi = np.percentile(boot[k], [2.5, 97.5])
                r[f'd_{k}'] = f'{point:+.3f} [{lo:+.3f}, {hi:+.3f}]'
            rows.append(r)
paired_df = pd.DataFrame(rows)
paired_df.to_csv(f'{RESULTS_DIR}/metrics/paired_vs_base.csv', index=False)
with pd.option_context('display.max_colwidth', None, 'display.width', 250):
    display(paired_df)

In [ ]:
# 13. Reliability diagrams (bars below the diagonal = overconfident)
for f in sorted(Path(RESULTS_DIR, 'metrics').glob('reliability_*.png')):
    print(f.name); display(Image(str(f), width=420))

In [ ]:
# 14. Make sure the Drive results are fully written before shutting down (skip this and the last few files might not upload if the runtime gets recycled)
from google.colab import drive
drive.flush_and_unmount()
print('all uploaded')

**How to read the results**

- The main result is **MedQA test set `direct`** (the main eval in the proposal); `cot` / `cot_vote8` are secondary.
- Accuracy: it only counts as a change if `net` clears `sig_line` or the `d_acc` interval doesn't include 0.
- Calibration: `d_conf` positive and `d_ece` positive (interval excludes 0) -> GRPO made the model more confident and worse calibrated; also check `hc_wrong_rate` (confidence >= 0.9 and wrong).
- Only call it a finding if both seeds point the same way; if only one seed is significant, say it's "not stable".
- MedMCQA / PubMedQA are transfer evals: the model was only trained on MedQA. The PubMedQA CoT prompt is slightly different (see the top); if `format_ok` is low, look at the generations first (`generations_*_pubmedqa_cot.jsonl`).